In [1]:
import os
import timeit

import numpy as np
import pandas as pd
import plotnine as p9
from IPython import get_ipython

import pyvinecopulib as pv

ipython = get_ipython()

# Display plots inline in Jupyter Notebook
ipython.run_line_magic("matplotlib", "inline")

# Minimal reporting for exception handlers
ipython.run_line_magic("xmode", "minimal")

# Automatically reload modules when they have changed
ipython.run_line_magic("load_ext", "autoreload")
ipython.run_line_magic("autoreload", "2")

# Display all output from a cell (not just the last line)
ipython.run_line_magic(
  "config", "InteractiveShell.ast_node_interactivity='all'"
)

# Set display options for pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)
pd.set_option("max_colwidth", None)
pd.set_option("display.precision", 3)

# Numpy settings
np.printoptions(
  precision=4,
  suppress=True,
  formatter={"float": "{:0.4f}".format},
  linewidth=80,
)

Exception reporting mode: Minimal


In [3]:
from sklearn.model_selection import train_test_split
from vinesforest.vines_forest import VinesForest

# Read data
n_vines = 50
seed = 12
num_threads = min(32, os.cpu_count() + 4)  # See default for concurrent.futures
data = pd.read_csv("data/daxreturns.csv")
if any(data < 0) or any(data > 1):
  data[data.columns] = pv.to_pseudo_obs(data.values)
controls = pv.FitControlsVinecop(family_set=[pv.tll], num_threads=num_threads)

data_train, data_test = train_test_split(
  data,
  test_size=0.4,
  # random_state=seed,
)

# Fit with Dissmann and forest
start = timeit.default_timer()
fitted_dissmann = pv.Vinecop.from_data(
  data_train.values,
  controls=controls,
)
loglik_dissmann_train = fitted_dissmann.loglik(data_train.values) / data_train.values.shape[0]
loglik_dissmann_test = fitted_dissmann.loglik(data_test.values)/ data_test.values.shape[0]
time_dissmann = timeit.default_timer() - start

start = timeit.default_timer()
controls.tree_algorithm = "sa"
fitted_sa = pv.Vinecop.from_data(
  data_train.values,
  controls=controls,
)
loglik_sa_train = fitted_sa.loglik(data_train.values) / data_train.values.shape[0]
loglik_sa_test = fitted_sa.loglik(data_test.values) / data_test.values.shape[0]
time_sa = timeit.default_timer() - start

start = timeit.default_timer()
controls.tree_algorithm = "mst_prim"
fitted_forest = VinesForest(
  n_vines=n_vines,
  controls=controls,
)
fitted_forest.fit(data_train.values)
loglik_forest_train = fitted_forest.loglik(data_train.values) / data_train.values.shape[0]
best_forest = np.argmax(loglik_forest_train)
loglik_forest_test = fitted_forest.loglik(data_test.values) / data_test.values.shape[0]
time_forest = timeit.default_timer() - start

indicators_train = loglik_forest_train > loglik_dissmann_train
indicators_test = loglik_forest_test > loglik_dissmann_test

# Dissman vs max forest in train vs test
logliks = [
  ["Dissmann", "Train", loglik_dissmann_train],
  ["Dissmann", "Test", loglik_dissmann_test],
  ["Forest", "Train", loglik_forest_train[best_forest]],
  ["Forest", "Test", loglik_forest_test[best_forest]],
  ["SA", "Train", loglik_sa_train],
  ["SA", "Test", loglik_sa_test]
]
df_logliks = pd.DataFrame(logliks, columns=["Method", "Set", "Loglik"])
print(df_logliks)

# Time
print(time_dissmann)
print(time_forest)
print(time_sa)

# Proportion of the forest that is better than the dissman
print(np.mean(indicators_train))
print(np.mean(indicators_test))

     Method    Set  Loglik
0  Dissmann  Train   6.459
1  Dissmann   Test   2.253
2    Forest  Train   6.466
3    Forest   Test   1.889
4        SA  Train   6.489
5        SA   Test   2.456
0.4375162549986271
20.862877517000015
3.382812648000254
0.06
0.28


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

indicators_train = loglik_forest_train > loglik_dissmann_train
indicators_test = loglik_forest_test > loglik_dissmann_test

confusion_matrix(indicators_test, indicators_train)
print(classification_report(indicators_test, indicators_train))

In [ ]:
df = pd.DataFrame(loglik_forest_test, columns=["loglik"])

p = (
  p9.ggplot(df, p9.aes(x="loglik"))
  + p9.geom_density()
  + p9.geom_vline(xintercept=loglik_dissmann_test, color="red")
)
p.show()

In [ ]:
fitted_dissmann.plot(
  tree=[0],
  vars_names=list(data.columns.str.replace(".DE", "")),
  add_edge_labels=False,
)

In [ ]:
# best_vine_train = np.argmax(fitted_forest.loglik(data_train.values))

In [ ]:
# np.argmax(fitted_forest.loglik(data_test.values))

In [ ]:
# best_vine.plot(
#   tree=[0],
#   vars_names=list(data.columns.str.replace(".DE", "")),
#   add_edge_labels=False,
# )